In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Path to dataset files: /kaggle/input/brain-tumor-mri-dataset


In [ ]:
# Verify image integrity & scan for corrupt files
import os
from PIL import Image

def clean_and_verify_dataset(base_path):
    valid_count = 0
    corrupt_count = 0
    non_image_count = 0

    print(f"Scanning dataset directory: {base_path}\n" + "="*50)

    for root, _, files in os.walk(base_path):
        for file in files:
            file_path = os.path.join(root, file)

            # Filter non-image extensions
            if not file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                non_image_count += 1
                continue

            try:
                with Image.open(file_path) as img:
                    img.verify()  # Verifies if file is readable and uncorrupted
                valid_count += 1
            except Exception as e:
                print(f"Corrupt image detected: {file_path} | Error: {e}")
                corrupt_count += 1
                # Remove corrupt file if path is writable
                try:
                    os.remove(file_path)
                    print(f"Successfully deleted corrupt file: {file_path}")
                except Exception:
                    print(f"File is in read-only path, skipping deletion.")

    print("\n" + "="*50)
    print(f"Verification Finished!")
    print(f"✔ Valid Images: {valid_count}")
    print(f"✖ Corrupt Images: {corrupt_count}")
    print(f"⚠ Non-image files skipped: {non_image_count}")

# Run cleaning process on kagglehub output path
clean_and_verify_dataset(path)

Scanning dataset directory: /kaggle/input/brain-tumor-mri-dataset

Verification Finished!
✔ Valid Images: 7200
✖ Corrupt Images: 0
⚠ Non-image files skipped: 0


In [ ]:
# ImageDataGenerator without manual rescaling
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_dir = os.path.join(path, 'Training')
test_dir = os.path.join(path, 'Testing')

# Data Augmentation (EfficientNet input expects 0-255 or preprocess_input)
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.15
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

class_names = list(train_generator.class_indices.keys())
print("\nClass labels detected:", class_names)

Found 4760 images belonging to 4 classes.
Found 840 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.

Class labels detected: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [ ]:
# Model Architecture with Fine-Tuning
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Unfreeze the top 20 layers for fine-tuning on MRI scans
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
outputs = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), # Lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 224, 224,  │          7 │ rescaling_2[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_3         │ (None, 224, 224,  │          0 │ normalization_1[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_3[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,219,175 (16.09 MB)

 Trainable params: 1,518,004 (5.79 MB)

 Non-trainable params: 2,701,171 (10.30 MB)

In [ ]:
# train Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

callbacks = [
    ModelCheckpoint('brain_tumor_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6, verbose=1)
]

history = model.fit(
    train_generator,
    epochs=12,
    validation_data=val_generator,
    callbacks=callbacks
)

Epoch 1/12
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 509ms/step - accuracy: 0.5587 - loss: 1.2825
Epoch 1: val_accuracy improved from None to 0.85952, saving model to brain_tumor_model.h5



Epoch 1: finished saving model to brain_tumor_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 122s 625ms/step - accuracy: 0.6792 - loss: 0.8969 - val_accuracy: 0.8595 - val_loss: 0.4869 - learning_rate: 1.0000e-04
Epoch 2/12
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 414ms/step - accuracy: 0.8237 - loss: 0.4777
Epoch 2: val_accuracy improved from 0.85952 to 0.89167, saving model to brain_tumor_model.h5



Epoch 2: finished saving model to brain_tumor_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 73s 493ms/step - accuracy: 0.8296 - loss: 0.4477 - val_accuracy: 0.8917 - val_loss: 0.3008 - learning_rate: 1.0000e-04
Epoch 3/12
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 418ms/step - accuracy: 0.8803 - loss: 0.3263
Epoch 3: val_accuracy improved from 0.89167 to 0.89643, saving model to brain_tumor_model.h5



Epoch 3: finished saving model to brain_tumor_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 73s 492ms/step - accuracy: 0.8746 - loss: 0.3395 - val_accuracy: 0.8964 - val_loss: 0.2407 - learning_rate: 1.0000e-04
Epoch 4/12
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 409ms/step - accuracy: 0.8957 - loss: 0.2896
Epoch 4: val_accuracy improved from 0.89643 to 0.91190, saving model to brain_tumor_model.h5



Epoch 4: finished saving model to brain_tumor_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 73s 488ms/step - accuracy: 0.8981 - loss: 0.2770 - val_accuracy: 0.9119 - val_loss: 0.2160 - learning_rate: 1.0000e-04
Epoch 5/12
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 427ms/step - accuracy: 0.9028 - loss: 0.2584
Epoch 5: val_accuracy improved from 0.91190 to 0.93095, saving model to brain_tumor_model.h5



Epoch 5: finished saving model to brain_tumor_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 76s 508ms/step - accuracy: 0.9071 - loss: 0.2481 - val_accuracy: 0.9310 - val_loss: 0.1743 - learning_rate: 1.0000e-04
Epoch 6/12
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 431ms/step - accuracy: 0.9106 - loss: 0.2329
Epoch 6: val_accuracy improved from 0.93095 to 0.93571, saving model to brain_tumor_model.h5



Epoch 6: finished saving model to brain_tumor_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 76s 509ms/step - accuracy: 0.9166 - loss: 0.2223 - val_accuracy: 0.9357 - val_loss: 0.1698 - learning_rate: 1.0000e-04
Epoch 7/12
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 421ms/step - accuracy: 0.9296 - loss: 0.1989
Epoch 7: val_accuracy did not improve from 0.93571
149/149 ━━━━━━━━━━━━━━━━━━━━ 74s 495ms/step - accuracy: 0.9269 - loss: 0.1935 - val_accuracy: 0.9298 - val_loss: 0.1711 - learning_rate: 1.0000e-04
Epoch 8/12
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 419ms/step - accuracy: 0.9282 - loss: 0.1763
Epoch 8: val_accuracy improved from 0.93571 to 0.95238, saving model to brain_tumor_model.h5



Epoch 8: finished saving model to brain_tumor_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 74s 498ms/step - accuracy: 0.9313 - loss: 0.1732 - val_accuracy: 0.9524 - val_loss: 0.1252 - learning_rate: 1.0000e-04
Epoch 9/12
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 418ms/step - accuracy: 0.9384 - loss: 0.1630
Epoch 9: val_accuracy did not improve from 0.95238
149/149 ━━━━━━━━━━━━━━━━━━━━ 83s 557ms/step - accuracy: 0.9393 - loss: 0.1590 - val_accuracy: 0.9524 - val_loss: 0.1315 - learning_rate: 1.0000e-04
Epoch 10/12
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 423ms/step - accuracy: 0.9532 - loss: 0.1337
Epoch 10: val_accuracy improved from 0.95238 to 0.95476, saving model to brain_tumor_model.h5



Epoch 10: finished saving model to brain_tumor_model.h5

Epoch 10: ReduceLROnPlateau reducing learning rate to 1.9999999494757503e-05.
149/149 ━━━━━━━━━━━━━━━━━━━━ 75s 502ms/step - accuracy: 0.9525 - loss: 0.1330 - val_accuracy: 0.9548 - val_loss: 0.1290 - learning_rate: 1.0000e-04
Epoch 11/12
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 420ms/step - accuracy: 0.9634 - loss: 0.1042
Epoch 11: val_accuracy improved from 0.95476 to 0.96190, saving model to brain_tumor_model.h5



Epoch 11: finished saving model to brain_tumor_model.h5
149/149 ━━━━━━━━━━━━━━━━━━━━ 74s 499ms/step - accuracy: 0.9582 - loss: 0.1090 - val_accuracy: 0.9619 - val_loss: 0.1187 - learning_rate: 2.0000e-05
Epoch 12/12
149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 420ms/step - accuracy: 0.9622 - loss: 0.1122
Epoch 12: val_accuracy did not improve from 0.96190
149/149 ━━━━━━━━━━━━━━━━━━━━ 74s 497ms/step - accuracy: 0.9582 - loss: 0.1173 - val_accuracy: 0.9524 - val_loss: 0.1253 - learning_rate: 2.0000e-05
Restoring model weights from the end of the best epoch: 11.


In [ ]:
# Test set evaluation
test_loss, test_acc = model.evaluate(test_generator)
print(f"\nUpdated Test Accuracy: {test_acc * 100:.2f}%")

50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 79ms/step - accuracy: 0.9175 - loss: 0.3840

Updated Test Accuracy: 91.75%


In [ ]:
# Download model in my PC
model.save('brain_tumor_model.h5')
files.download('brain_tumor_model.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>